# simple_tutor — component bench

Poke the tutor one component at a time. Companion to `memory/simple_tutor_audit.md`.

Layered by what each part needs, cheapest first:

| § | layer | needs |
|---|---|---|
| 1 | setup | Django |
| 2 | **grader, deterministic** | **nothing** — no DB, no network |
| 3 | prompt assembly | nothing |
| 4 | engine text helpers | nothing |
| 5 | DB components | a session row |
| 6 | LLM components | Ollama / cloud key |
| 7 | full turn | Ollama + a session |

§2–4 are 109 of the package's 169 functions and run instantly. Start there.

Kernel: the project venv (`venv/bin/python`).

## 1 — Setup

Run once. Everything below depends on it.

In [ ]:
import os, sys, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / 'manage.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'ai_tutor.config.settings')

# Production uses the compact Block-0. Without this you silently measure the
# FULL template (20475 chars vs 13500) — a real bug that has bitten twice.
os.environ.setdefault('QWEN_BLOCK_0', 'compact')

import django; django.setup()
print('root:', ROOT)
print('django ok')

## 2 — Grader, deterministic tiers

`grade_answer` is duck-typed. Any object with the right attributes works, so a
`SimpleNamespace` is enough — no fixtures, no migrations.

```
question_type   correct_answer   answer_data   option_a..option_d   question_text   pk
```

This is the highest-value thing to test by hand: a wrong verdict here silently
corrupts every downstream guard.

In [ ]:
from types import SimpleNamespace
from apps.tutoring.simple_tutor.grader import grade_answer

def Q(**kw):
    """Minimal question stub. Override any field per call."""
    base = dict(pk=0, question_type='mcq', correct_answer='', answer_data={},
                question_text='', option_a='', option_b='', option_c='', option_d='')
    base.update(kw)
    return SimpleNamespace(**base)

def show(q, answer):
    r = grade_answer(question=q, student_answer=answer)
    print(f"  {answer!r:<28} -> {r.verdict.value:<9} tier={r.tier:<12} conf={r.confidence:.2f}")
    return r

print('helpers ready')

### 2a — MCQ

Edit the answers list. Worth probing: bare letters, `Option B`, numbered
options, the option text typed out, a distinctive substring, and a value that
matches an option numerically.

In [ ]:
q = Q(question_type='mcq', correct_answer='C',
      question_text='Which figure is the easting on a four-figure grid reference?',
      option_a='The last two digits', option_b='The northing',
      option_c='The first two digits', option_d='The scale')

for a in ['C', 'c', 'C.', 'Option C', '3', 'the first two digits',
          'first two digits', 'easting', 'B', 'I think C but maybe A']:
    show(q, a)

### 2b — Numeric / math

`short_numeric` is the production type for computed answers. `answer_data`
carries `{computed, model_answer, unit, parameters}`. Try units, spelled-out
numbers, and near-misses to see where tolerance sits.

In [ ]:
q = Q(question_type='short_numeric', correct_answer='1.8',
      question_text='On a 1:30,000 map two villages are 6 cm apart. Real distance in km?',
      answer_data={'computed': 1.8, 'unit': 'km'})

for a in ['1.8', '1.8 km', '1,8', 'one point eight', '1.80', '1.79', '2', '18']:
    show(q, a)

### 2c — Extractors in isolation

When a verdict surprises you, drop a level and see what was parsed.

In [ ]:
from apps.tutoring.simple_tutor.grader import (
    _extract_mcq_letter, _extract_option_text_match, _extract_last_number,
    _spoken_to_numeric, _norm_option, _option_number,
)

for s in ['B', 'Option B', 'answer: b', '2', 'the northing', 'twelve point five']:
    print(f"  {s!r:<22} letter={_extract_mcq_letter(s)!r:<6} "
          f"num={_extract_last_number(s)!r:<8} spoken={_spoken_to_numeric(s)!r}")

print('\noption text match against the §2a question:')
q = Q(correct_answer='C', option_a='The last two digits', option_b='The northing',
      option_c='The first two digits', option_d='The scale')
for s in ['easting', 'northing', 'the scale', 'digits']:
    print(f"  {s!r:<16} -> {_extract_option_text_match(q, s)!r}")

### 2d — Sweep your own cases

Add rows as `(question, answer, expected)`. Anything that disagrees prints as
`MISMATCH` — this is the cheapest regression harness in the package.

In [ ]:
mcq = Q(question_type='mcq', correct_answer='C',
        option_a='The last two digits', option_b='The northing',
        option_c='The first two digits', option_d='The scale')
num = Q(question_type='short_numeric', correct_answer='1.8',
        answer_data={'computed': 1.8, 'unit': 'km'})

CASES = [
    (mcq, 'C',                    'correct'),
    (mcq, 'the first two digits', 'correct'),
    (mcq, 'B',                    'incorrect'),
    (num, '1.8 km',               'correct'),
    (num, '2',                    'incorrect'),
]

bad = 0
for q_, a_, exp in CASES:
    got = grade_answer(question=q_, student_answer=a_).verdict.value
    flag = '' if got == exp else '   <-- MISMATCH'
    if flag: bad += 1
    print(f"  {a_!r:<26} expected={exp:<10} got={got:<10}{flag}")
print(f"\n{len(CASES)-bad}/{len(CASES)} as expected")

## 3 — Prompt assembly

Pure. Inspect exactly what the model is told, per family.

**Pin `QWEN_BLOCK_0`** (§1 does it) or you read the full template while
production ships the compact one.

In [ ]:
import json
from apps.tutoring.simple_tutor.family_prompts import build_family_block_0
from apps.tutoring.simple_tutor.prompts import TOOL_SCHEMAS, _BLOCK_0_TEMPLATE

# build_family_block_0(family, base_template) — prompts.py owns the base and
# passes it in, so family=None/anthropic returns the base UNCHANGED. Pass the
# real template or the None row reads as "4 chars".
for fam in ['qwen', 'gemini', 'gemma', None]:
    n = len(build_family_block_0(fam, _BLOCK_0_TEMPLATE))
    note = '  (= base, unchanged)' if fam in (None, 'anthropic') else ''
    print(f"  family={str(fam):<8} block_0 = {n:>6} chars{note}")
print(f"  {'base':<15} = {len(_BLOCK_0_TEMPLATE):>6} chars")

print(f"\ntool schemas: {len(TOOL_SCHEMAS)} tools, "
      f"{sum(len(t['description']) + len(json.dumps(t['input_schema'])) for t in TOOL_SCHEMAS)} bytes")
for t in TOOL_SCHEMAS:
    props = t['input_schema'].get('properties', {})
    longest = max((len(v.get('description', '')) for v in props.values()), default=0)
    print(f"  {t['name']:<20} desc={len(t['description']):>5}  "
          f"longest_param={longest:>4}  params={list(props)}")

In [ ]:
# Read the actual Block-0 the qwen tutor receives.
block0 = build_family_block_0('qwen', 'BASE')
print(block0[:3000])
print(f"\n... [{len(block0)} chars total]")

In [ ]:
# Which tools does Block-0 actually explain? A tool named 0x is one the model
# only meets through its schema description.
for name in [t['name'] for t in TOOL_SCHEMAS]:
    print(f"  {name:<20} mentioned {block0.count(name)}x in Block-0")

## 4 — Engine text helpers

Pure post-processing applied to every reply before the student sees it. These
are where a reply gets silently rewritten, so worth reading directly.

In [ ]:
from apps.tutoring.simple_tutor.engine import (
    _scrub_engine_vocab, _strip_trailing_prose_question,
    _contains_lettered_options, _looks_like_question_para, _call_mode,
)
from apps.tutoring.simple_tutor.stream_filter import safe_cut_index

sample = ("Nice work. I'll call record_answer(extracted_answer='C') now.\n\n"
          "A) north   B) south   C) east   D) west\n\nWhat comes next?")
print('scrubbed:\n', _scrub_engine_vocab(sample))
print('\nhas lettered options:', _contains_lettered_options(sample))
print('trailing question stripped:\n', _strip_trailing_prose_question(sample))
print('\ncall_mode qwen =', _call_mode('qwen'), ' anthropic =', _call_mode('anthropic'))
partial = 'The answer is |||MED'
print('safe_cut_index:', safe_cut_index(partial), 'of', len(partial))

## 5 — DB components

Needs a real session. **Read-only below** — nothing here writes.
The mutating handlers are in §5c, commented out.

In [ ]:
from apps.tutoring.models import TutorSession

s = TutorSession.objects.order_by('-id').first()
if s is None:
    print('no sessions in this DB — skip §5')
else:
    print(f"session {s.id}  lesson={getattr(s.lesson, 'id', None)}  "
          f"step={s.current_step_index}  status={s.status}")
    # current_question_id / _source are the SERVER-side anchor fields. They are
    # still on the model but nothing writes them — pick_current_question was
    # dropped in 2afc4e5. Expect None; see memory/simple_tutor_audit.md §4.
    print('current_question_id :', s.current_question_id,
          ' source:', s.current_question_source)
    es = s.engine_state or {}
    print('engine_state keys   :', list(es))
    print('forced_advances     :', es.get('forced_advances'))
    print('answered_correct    :', es.get('answered_correct'))

In [ ]:
# 5b — the question pool the LLM is shown for this step (read-only)
#
# An EMPTY pool is normal for two reasons, both worth checking before you
# suspect a bug:
#   1. current_step_index is past the last step (session finished).
#   2. TUTORING_QUESTION_TYPES filters the pool. It defaults to mcq-ONLY, so
#      short_answer / short_numeric questions are excluded by construction —
#      see memory/simple_tutor_audit.md §5b.
from apps.tutoring.simple_tutor.tools import build_question_pool, _allowed_tutoring_types

SESSION_ID = None   # set to pin a specific session

if SESSION_ID:
    s = TutorSession.objects.get(id=SESSION_ID)

if s is not None:
    n_steps = s.lesson.steps.count() if s.lesson_id else 0
    past_end = s.current_step_index >= n_steps
    print(f"session {s.id}  step {s.current_step_index} of {n_steps}"
          f"{'   <- past last step, pool will be empty' if past_end else ''}")
    print('allowed question types:', _allowed_tutoring_types(),
          '  (default is mcq-only)\n')

    pool = build_question_pool(s)
    print(f"{len(pool)} questions in pool\n")
    for i, q_ in enumerate(pool, 1):
        print(f"  {i}. [{getattr(q_, 'question_type', '?')}] "
              f"{str(getattr(q_, 'question_text', ''))[:88]}")
        print(f"      correct={getattr(q_, 'correct_answer', None)!r}")

    if not pool:
        # Sessions still mid-lesson: an in-flight question means one is posed
        # and awaiting an answer, which is the interesting state to inspect.
        cand = (TutorSession.objects
                .filter(in_flight_question__isnull=False)
                .order_by('-id')
                .values_list('id', 'current_step_index')[:8])
        print('\nsessions with a live question — set SESSION_ID to one of:',
              list(cand) or '(none)')

In [ ]:
# 5c — MUTATING. Uncomment deliberately; these write to the session.
#
# from apps.tutoring.simple_tutor.tools import (
#     maybe_advance_step, autograde_bare_answer_if_clear,
# )
# print(maybe_advance_step(s))
# print(autograde_bare_answer_if_clear(session=s, user_input='C'))
print('§5c is opt-in — read the handlers before running them')

In [ ]:
# 5d — which model would this student get, and therefore which prompt family?
from apps.tutoring.simple_tutor.model_choice import resolve_for_session
from apps.llm.model_profiles import get_model_profile

if s is not None:
    cfg = resolve_for_session(s)
    print('resolved config:', cfg)
    if cfg is not None:
        spec = f'{cfg.provider}/{cfg.model_name}'
        prof = get_model_profile(spec)
        print(f"  spec={spec}  family={getattr(prof, 'family', None)}")

## 6 — LLM components

Embeddings are local (ONNX MiniLM, no network). The verifier tier needs a judge
provider; offline it falls back to `_local_verifier_chain`.

In [ ]:
# 6a — embedding gate (local, no network)
from apps.tutoring.simple_tutor.grader import _grade_embedding_gate

q = Q(question_type='short_answer',
      correct_answer='The easting is read first, then the northing.',
      question_text='In what order do you read a four-figure grid reference?')

for a in ['easting first, then northing',
          'you read the northing before the easting',
          'left to right then bottom to top',
          'no idea']:
    r = _grade_embedding_gate(q, a)
    print(f"  {a!r:<46} -> {r.verdict.value if r else 'MIDDLE BAND -> verifier LLM'}")

In [ ]:
# 6b — full short_answer path (may call the verifier LLM)
for a in ['easting first, then northing', 'northing then easting']:
    show(q, a)

## 7 — A full turn

The end-to-end path. Slow (~10–15 s/turn on qwen3-4b) and it **writes turns to
the session**, so use a throwaway one.

`scripts/measure_call_compliance.py` is the batch version of this and reports
per-tool compliance — prefer it for anything you intend to quote as a number.
Note its default persona is `error_prone`, which never answers correctly; use
`--persona capable` to exercise a correct-answering student.

In [ ]:
# Uncomment to drive one real turn.
#
# os.environ['TUTOR_MODEL_OVERRIDE'] = 'local_ollama/qwen3-4b-jetson'
# from apps.tutoring.simple_tutor.engine import respond_for_view
# out = respond_for_view(s, 'C')
# print(out.get('reply'))
# print('\ntools:', out.get('tools_called'), ' verdict:', out.get('verdict'))
print('§7 is opt-in — it writes turns')